In [67]:
import os
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping


print("Tensorflow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


Tensorflow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [68]:
df = pd.read_csv('/content/sample_data/ecommerce_complaints_250k_unique.csv')

df.rename(columns={'complaint_text': 'complaint'}, inplace=True)
print('Dataset shape', df.shape)
print('\n columns:')
print(df.columns.tolist())

df.head()


Dataset shape (250000, 3)

 columns:
['complaint_id', 'complaint', 'department']


,complaint_id,complaint,department
0,C038684,An unknown device accessed my account and I ca...,Account & Security
1,C064940,My support request remains unanswered after se...,Customer Support
2,C003955,My registered email was changed despite upload...,Account & Security
3,C120375,My shipment shows a delivery exception while t...,Order & Delivery
4,C172862,The product quality is not advertised after I ...,Product & Seller


In [69]:
#Basic validation

print("missing values\n")
print(df.isnull().sum())
print('\n Duplicate',df.duplicated().sum())
print('\n Deparment Distribution:\n')
print(df['department'].value_counts())

missing values

complaint_id    0
complaint       0
department      0
dtype: int64

 Duplicate 0

 Deparment Distribution:

department
Account & Security       41667
Customer Support         41667
Order & Delivery         41667
Payments & Refunds       41667
Product & Seller         41666
Returns & Replacement    41666
Name: count, dtype: int64


In [70]:
print("\nUnique departments:")
print(df["department"].unique())


Unique departments:
['Account & Security' 'Customer Support' 'Order & Delivery'
 'Product & Seller' 'Payments & Refunds' 'Returns & Replacement']


In [72]:
#Text Cleaning
df["complaint"] = (
    df["complaint"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["complaint"].head()

,complaint
0,an unknown device accessed my account and i ca...
1,my support request remains unanswered after se...
2,my registered email was changed despite upload...
3,my shipment shows a delivery exception while t...
4,the product quality is not advertised after i ...


In [73]:
#NLP Preprocessing

X = df['complaint']
y = df['department']

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExample complaint:")
print(X.iloc[0])

print("\nExample label:")
print(y.iloc[0])

print('\nMaxLen of Complaint')
print(df['complaint'].str.len().max())

X shape: (250000,)
y shape: (250000,)

Example complaint:
an unknown device accessed my account and i cannot reverse it, and the available instructions do not help. please restore my secure access, because the automated recovery process failed.

Example label:
Account & Security

MaxLen of Complaint
241


In [74]:
#Label Encoding

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print('Department mapping:')

for index, deparment in enumerate(label_encoder.classes_):
  print(f'{index} -> {deparment}')

Department mapping:
0 -> Account & Security
1 -> Customer Support
2 -> Order & Delivery
3 -> Payments & Refunds
4 -> Product & Seller
5 -> Returns & Replacement


In [96]:
#Train test Split

# First split: 80% training, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

# Second split: 10% validation, 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print(len(X_train), len(y_train))
print(len(X_val), len(y_val))
print(len(X_test), len(y_test))

Training: 200000
Validation: 25000
Testing: 25000
200000 200000
25000 25000
25000 25000


In [76]:
#Verify class distribution

print("Training distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation distribution:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest distribution:")
print(pd.Series(y_test).value_counts().sort_index())

Training distribution:
0    33333
1    33334
2    33334
3    33333
4    33333
5    33333
Name: count, dtype: int64

Validation distribution:
0    4167
1    4166
2    4167
3    4167
4    4166
5    4167
Name: count, dtype: int64

Test distribution:
0    4167
1    4167
2    4166
3    4167
4    4167
5    4166
Name: count, dtype: int64


In [77]:
#Tokenizetion

tokenizer = Tokenizer(
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index) + 1


print("Vocabulary size:", vocab_size)

Vocabulary size: 425


In [79]:
#Vocab inspection
print("First 20 vocabulary words:")

for word, index in list(tokenizer.word_index.items())[:20]:
    print(index, "->", word)
print("\nOOV token index:")
print(tokenizer.word_index.get("<OOV>"))

First 20 vocabulary words:
1 -> <OOV>
2 -> the
3 -> i
4 -> please
5 -> my
6 -> and
7 -> because
8 -> was
9 -> a
10 -> after
11 -> delivery
12 -> is
13 -> has
14 -> need
15 -> return
16 -> cannot
17 -> account
18 -> so
19 -> can
20 -> despite

OOV token index:
1


In [80]:
#Convert Text into Sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print('Orignal:')
print(X_train.iloc[0])

print('\nSequence:')
print(X_train_seq[0])

print(tokenizer.word_index)

Orignal:
the received item differs from the photos after i checked the stated specifications. and i cannot trust the listing. please review the seller's representation. because i saved the original listing.

Sequence:
[2, 88, 26, 203, 82, 2, 411, 10, 3, 93, 2, 69, 107, 6, 3, 16, 218, 2, 29, 4, 32, 2, 71, 83, 7, 3, 226, 2, 47, 29]
{'<OOV>': 1, 'the': 2, 'i': 3, 'please': 4, 'my': 5, 'and': 6, 'because': 7, 'was': 8, 'a': 9, 'after': 10, 'delivery': 11, 'is': 12, 'has': 13, 'need': 14, 'return': 15, 'cannot': 16, 'account': 17, 'so': 18, 'can': 19, 'despite': 20, 'product': 21, 'remains': 22, 'without': 23, 'support': 24, 'not': 25, 'item': 26, 'process': 27, 'while': 28, 'listing': 29, 'which': 30, 'payment': 31, 'review': 32, 'information': 33, 'status': 34, 'provide': 35, 'order': 36, 'details': 37, 'leaving': 38, 'to': 39, 'correct': 40, 'seller': 41, 'replacement': 42, 'package': 43, 'investigate': 44, 'unresolved': 45, 'this': 46, 'original': 47, 'me': 48, 'transaction': 49, 'case'

In [84]:
#Determine sequence length
train_lengths = [len(sequence) for sequence in X_train_seq]

print("Minimum:", np.min(train_lengths))
print("Maximum:", np.max(train_lengths))
print("Mean:", np.mean(train_lengths))
print("95th percentile:", np.percentile(train_lengths, 95))
print("99th percentile:", np.percentile(train_lengths, 99))

MAX_LEN = int(np.percentile(train_lengths, 95))

print("MAX_LEN:", MAX_LEN)


Minimum: 24
Maximum: 30
Mean: 27.64318
95th percentile: 30.0
99th percentile: 30.0
MAX_LEN: 30


In [85]:
#Padding

# X = df['complaint']
# y = pd.get_dummies(df['department']).values

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen = MAX_LEN,
    padding = 'post',
    truncating = 'post'
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen = MAX_LEN,
    padding = 'post',
    truncating = 'post'
)

print("Training shape:",X_train_pad.shape)
print("X_val_pad:", X_val_pad.shape)
print("Testing shape:",X_test_pad.shape)

Training shape: (200000, 30)
X_val_pad: (25000, 30)
Testing shape: (25000, 30)


In [87]:
#Example of padding
print("Sequence:")
print(X_train_seq[5])

print("\nPadded:")
print(X_train_pad[5])


Sequence:
[373, 374, 62, 375, 5, 171, 10, 204, 196, 114, 349, 4, 35, 165, 254, 181, 7, 30, 133, 2, 27, 299, 300, 3, 14, 261, 39, 146, 161]

Padded:
[373 374  62 375   5 171  10 204 196 114 349   4  35 165 254 181   7  30
 133   2  27 299 300   3  14 261  39 146 161   0]


In [91]:
y_train.shape

(200000,)

In [88]:
#Model building

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=100,
        input_length= MAX_LEN
    )
)

model.add(
    LSTM(
        32,
        dropout=0.2,
        return_sequences=False
    )
)

model.add(Dense(16, activation='relu'))

model.add(Dense(len(label_encoder.classes_), activation='softmax'))

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [93]:
#Compile Model

model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)


In [95]:
print("X_train_pad:", X_train_pad.shape)
print("y_train:", y_train.shape)

print("X_val_pad:", X_val_pad.shape)
print("y_val:", y_val.shape)

print("X_test_pad:", X_test_pad.shape)
print("y_test:", y_test.shape)

X_train_pad: (200000, 30)
y_train: (200000,)
X_val_pad: (25000, 30)
y_val: (25000,)
X_test_pad: (25000, 30)
y_test: (25000,)


In [94]:
#Model Training

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    min_delta=0.1,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_data = (X_train_pad, y_val),
    epochs=10,
    batch_size = 60,
    callbacks = [early_stopping],
    verbose = 1
)

Epoch 1/10
3332/3334 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9481 - loss: 0.1476

ValueError: Data cardinality is ambiguous. Make sure all arrays contain the same number of samples.'x' sizes: 200000
'y' sizes: 25000


In [33]:
#Evaluation
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=1
)

print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - accuracy: 0.1667 - loss: 1.7918

Test Loss: 1.7917659282684326
Test Accuracy: 0.16665999591350555


In [36]:
# Debug

for department in label_encoder.classes_:
    print("\n" + "=" * 60)
    print(department)
    print("=" * 60)

    samples = df[df["department"] == department]["complaint"].head(3)

    for text in samples:
        print("-", text)


Account & Security
- The same issue keeps returning after each attempt. Normal login attempts triggered an unexpected security restriction. The latest update shows this problem. Please address the underlying issue and confirm what action will be taken.
- The same issue keeps returning after each attempt. The latest issue is that the recovery process is sending verification requests to unusable contact details. This situation needs attention. A practical resolution is needed instead of another generic update.
- The problem is still active despite the steps already completed. A security review is preventing normal account access. Something in the recent process has gone wrong. The problem should be resolved using the actual transaction or case history.

Customer Support
- The main issue is this. The available information does not provide a reliable resolution. The concern is that an earlier support request has not received a meaningful resolution. Please investigate the actual cause and

In [37]:
print("Vocabulary size:", len(tokenizer.word_index))

print("\nSample vocabulary:")
print(list(tokenizer.word_index.items())[:50])

Vocabulary size: 331

Sample vocabulary:
[('<OOV>', 1), ('the', 2), ('a', 3), ('is', 4), ('issue', 5), ('has', 6), ('not', 7), ('please', 8), ('problem', 9), ('and', 10), ('after', 11), ('what', 12), ('same', 13), ('case', 14), ('update', 15), ('latest', 16), ('information', 17), ('be', 18), ('current', 19), ('to', 20), ('process', 21), ('this', 22), ('resolution', 23), ('status', 24), ('that', 25), ('provide', 26), ('request', 27), ('does', 28), ('records', 29), ('available', 30), ('review', 31), ('have', 32), ('an', 33), ('of', 34), ('unresolved', 35), ('situation', 36), ('earlier', 37), ('concern', 38), ('was', 39), ('actual', 40), ('been', 41), ('happened', 42), ('needs', 43), ('clear', 44), ('needed', 45), ('explain', 46), ('without', 47), ('remains', 48), ('checking', 49), ('reviewed', 50)]
